In [1]:
#Ref
import pandas as pd
import glob
import numpy as np
import plotly.express as px
import re

path = '/Users/jiakai/Desktop/SURF/code/_spirit/'
# n_cycles = 120
dim = 10
gamma = 1e-07


# Pattern to match all relevant CSV files
file_pattern = path + f"negative_field_cycle_v2_dim{dim}_anisotropy1.5_ncycles*_gamma1e-07_H_high6.0_H_low*.csv"

# List all matching files
csv_files = glob.glob(file_pattern)

for f in csv_files:
    print(f)

# Extract H_low values
H_lows = []
for path in csv_files:
    match = re.search(r'H_low([0-9.]+)\.csv', path)
    if match:
        h_low = float(match.group(1))
        H_lows.append(h_low)

print(H_lows)

/Users/jiakai/Desktop/SURF/code/_spirit/negative_field_cycle_v2_dim10_anisotropy1.5_ncycles240_gamma1e-07_H_high6.0_H_low3.0.csv
[3.0]


In [3]:
#Correlations
path = '/Users/jiakai/Desktop/SURF/code/_spirit/'
# Pattern to match all relevant CSV files
file_pattern = path + f"decay_correlations_negative_field_cycle_v2_dim{dim}_anisotropy1.5_ncycles*_gamma1e-07_H_high6.0_H_low*.csv"
#decay_correlations_negative_field_cycle_dim10_anisotropy1.5_ncycles120_gamma1e-07_H_high6.0_H_low3.0
# decay_correlations_negative_field_cycle_v2_dim4_anisotropy1.5_ncycles1200_gamma1e-07_H_high6.0_H_low4.0
# List all matching files
csv_files = glob.glob(file_pattern)
# /Users/jiakai/Desktop/SURF/code/_spirit/decay_correlations_negative_field_cycle_dim4_anisotropy0.7_ncycles1200_gamma1e-06_H_high3.0_H_low0.5.csv
for f in csv_files:
    print(f)

# Extract H_low values
H_lows = []
for path in csv_files:
    match = re.search(r'H_low([0-9.]+)\.csv', path)
    if match:
        h_low = float(match.group(1))
        H_lows.append(h_low)

print(H_lows)


/Users/jiakai/Desktop/SURF/code/_spirit/decay_correlations_negative_field_cycle_v2_dim10_anisotropy1.5_ncycles120_gamma1e-07_H_high6.0_H_low3.0.csv
[3.0]


In [8]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import pandas as pd
import plotly.express as px

path = '/Users/jiakai/Desktop/SURF/code/_spirit/'
n_cycles = 120
H_low_values = [3.0]

# Pick a distinct color for each H_low
color_map = {H: px.colors.qualitative.Plotly[i] for i, H in enumerate(H_low_values)}

# Create stacked subplots with shared x-axis alignment
fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.05,
    subplot_titles=("Susceptibility χ", "Average Correlation")
)

for H_low in H_low_values:
    # File paths
    #negative_field_cycle_v2_dim4_anisotropy1.5_ncycles1200_gamma1e-07_H_high6.0_H_low5.0
    path_chi = path + f"negative_field_cycle_v2_dim{dim}_anisotropy1.5_ncycles240_gamma1e-07_H_high6.0_H_low{H_low}.csv"
    # decay_correlations_negative_field_cycle_v2_dim4_anisotropy1.5_ncycles1200_gamma1e-07_H_high6.0_H_low6.0
    path_corr = path + f"decay_correlations_negative_field_cycle_v2_dim{dim}_anisotropy1.5_ncycles120_gamma1e-07_H_high6.0_H_low{H_low}.csv"

    # Read CSVs
    df_chi = pd.read_csv(path_chi)
    df_corr = pd.read_csv(path_corr)

    # Group chi data
    df_chi_avg = df_chi.groupby(['i', 'Ht']).agg(
        chi_mean=('chi', 'mean'),
        chi_std=('chi', 'std')
    ).reset_index()
    df_chi_avg['chi_sem'] = df_chi_avg['chi_std'] / (n_cycles ** 0.5)

    # Group correlation data
    df_corr_avg = df_corr.groupby("k", as_index=False).agg(
        corr_avg=("corr", "mean"),
        corr_sem=("corr", lambda x: x.std(ddof=1) / (len(x) ** 0.5))
    )

    # Susceptibility (top) — same color as correlation for same H_low
    fig.add_trace(
        go.Scatter(
            x=df_chi_avg["i"],
            y=df_chi_avg["chi_mean"],
            error_y=dict(type="data", array=df_chi_avg["chi_sem"], visible=True),
            mode='lines+markers',
            marker=dict(color=color_map[H_low]),
            line=dict(color=color_map[H_low]),
            name=f"Chi H_low={H_low}"
        ),
        row=1, col=1
    )

    # Correlation (bottom) — same color as chi
    fig.add_trace(
        go.Scatter(
            x=df_corr_avg["k"],
            y=df_corr_avg["corr_avg"],
            error_y=dict(type='data', array=df_corr_avg["corr_sem"], visible=True),
            mode='lines+markers',
            marker=dict(color=color_map[H_low]),
            line=dict(color=color_map[H_low]),
            name=f"Correlation H_low={H_low}"
        ),
        row=2, col=1
    )

# Axis labels
fig.update_xaxes(title_text="MCS Step", row=2, col=1)
fig.update_yaxes(title_text="Susceptibility", row=1, col=1)
fig.update_yaxes(title_text="Average Correlation", row=2, col=1)

# Layout
fig.update_layout(
    # height=800,
    title="Susceptibility and Correlation vs Field",
    template="plotly_white",
    legend=dict(x=0.02, y=0.02)
)

# Save
fig.write_html(f"stacked_susceptibility_correlation_v2_dim{dim}.html")


In [9]:
fig.show()